# Multi-Source OOD Computation — M0/M2

Build pair-specific diagnostics for assumed models M0/M2. Per-model log marginal
likelihood, summary-distance, and posterior diagnostics are reused from the four-model
cache. Model probabilities, PMP errors, global-surprise flags, and summary ambiguity
are recomputed within this two-model comparison and saved under
`results/model_pairs/m0_m2`.


In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from scipy.special import softmax

PROJECT_DIR = Path("/Users/yimingzang/Documents/Project/benchmark2")
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from benchmark.examples.diffusion.config import TrainingConfig
from benchmark.examples.diffusion.results.multisource_pipeline import load_cached_all_observed
from benchmark.examples.diffusion.results.observed_datasets import OBSERVED_DATASETS

PAIR_MODELS = ('m0', 'm2')
PAIR_KEY = "m0_m2"
DIFFUSION_DIR = PROJECT_DIR / "benchmark" / "examples" / "diffusion"
PAIR_RESULT_DIR = DIFFUSION_DIR / "results" / "model_pairs" / PAIR_KEY


INFO:jax._src.xla_bridge:Unable to initialize backend 'rocm': module 'jaxlib.xla_extension' has no attribute 'GpuAllocatorConfig'
INFO:jax._src.xla_bridge:Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: dlopen(libtpu.so, 0x0001): tried: 'libtpu.so' (no such file), '/System/Volumes/Preboot/Cryptexes/OSlibtpu.so' (no such file), '/opt/anaconda3/envs/benchmark2/bin/../lib/libtpu.so' (no such file), '/usr/lib/libtpu.so' (no such file, not in dyld cache), 'libtpu.so' (no such file), '/usr/local/lib/libtpu.so' (no such file), '/usr/lib/libtpu.so' (no such file, not in dyld cache)
INFO:bayesflow:Using backend 'jax'


## Configuration


In [2]:
summary_configs = {
    "S=D": TrainingConfig(summary_multiplier=1),
    "S=2D": TrainingConfig(summary_multiplier=2),
    "S=4D": TrainingConfig(summary_multiplier=4),
    "S=6D": TrainingConfig(summary_multiplier=6),
}

metric = "l2"
recompute = False

print("Assumed models:", PAIR_MODELS)
print("Pair result directory:", PAIR_RESULT_DIR)
OBSERVED_DATASETS


Assumed models: ('m0', 'm2')
Pair result directory: /Users/yimingzang/Documents/Project/benchmark2/benchmark/examples/diffusion/results/model_pairs/m0_m2


('empirical',
 'simulated_from_m0',
 'simulated_from_m1',
 'simulated_from_m2',
 'simulated_from_m3',
 'm3_fast_30',
 'm3_slow_30',
 'm3_fast_slow_30')

## Pairwise Cache Helpers


In [3]:
def pair_paths(tag, metric="l2"):
    suffix = "" if metric == "l2" else f"_{metric}"
    return {
        "results": PAIR_RESULT_DIR / f"npe_{tag}_all_observed_results_gold.csv",
        "diagnostic": PAIR_RESULT_DIR / f"npe_{tag}_all_observed_diagnostic{suffix}.csv",
        "posterior": PAIR_RESULT_DIR / "posterior_diagnostics" / f"npe_{tag}_all_observed_posterior.csv",
        "posterior_plot": PAIR_RESULT_DIR / "posterior_diagnostics" / f"npe_{tag}_all_observed_posterior_plot{suffix}.csv",
    }


def load_pair_frames(config, metric="l2"):
    paths = pair_paths(config.summary_label, metric=metric)
    missing = [str(path) for path in paths.values() if not path.exists()]
    if missing:
        raise FileNotFoundError("\n".join(missing))
    return {name: pd.read_csv(path, keep_default_na=False) for name, path in paths.items()}


def pairwise_results(full_results):
    keys = ["dataset", "id"]
    logml_cols = [f"log_ml_{model}" for model in PAIR_MODELS]
    gold_logml_cols = [f"gold_log_ml_{model}" for model in PAIR_MODELS]
    ess_cols = [f"importance_ess_{model}" for model in PAIR_MODELS]
    output = full_results[keys + logml_cols + ess_cols + gold_logml_cols].copy()

    npe_pmp = softmax(output[logml_cols].to_numpy(dtype=np.float64), axis=1)
    gold_pmp = softmax(output[gold_logml_cols].to_numpy(dtype=np.float64), axis=1)
    for index, model in enumerate(PAIR_MODELS):
        output[f"pmp_{model}"] = npe_pmp[:, index]
        output[f"gold_pmp_{model}"] = gold_pmp[:, index]
        output[f"signed_logml_error_{model}"] = output[f"log_ml_{model}"] - output[f"gold_log_ml_{model}"]
        output[f"signed_pmp_error_{model}"] = output[f"pmp_{model}"] - output[f"gold_pmp_{model}"]
        output[f"abs_pmp_error_{model}"] = output[f"signed_pmp_error_{model}"].abs()
    return output


def pairwise_diagnostic(results, full_diagnostic, eps=1e-8):
    diagnostic_cols = [
        column
        for model in PAIR_MODELS
        for column in (
            f"d_{model}",
            f"rho_{model}",
            f"dm_low_{model}",
            f"dm_high_{model}",
            f"regime_{model}",
        )
    ]
    output = results.merge(
        full_diagnostic[["dataset", "id"] + diagnostic_cols],
        on=["dataset", "id"],
        how="left",
        validate="one_to_one",
    )

    distance_matrix = output[[f"d_{model}" for model in PAIR_MODELS]].to_numpy(dtype=np.float64)
    order = np.argsort(distance_matrix, axis=1)
    output["closest_summary_model"] = [PAIR_MODELS[index] for index in order[:, 0]]
    output["d_min"] = distance_matrix[np.arange(len(output)), order[:, 0]]
    output["d_second"] = distance_matrix[np.arange(len(output)), order[:, 1]]
    output["summary_ambiguity_true"] = 1.0 / (np.abs(output["d_second"] - output["d_min"]) + eps)
    regime_cols = [f"regime_{model}" for model in PAIR_MODELS]
    output["globally_high_surprise"] = output[regime_cols].eq("high surprise").all(axis=1)
    output["at_least_one_not_high_surprise"] = ~output["globally_high_surprise"]
    output["summary_ambiguity"] = np.where(
        output["globally_high_surprise"],
        output["summary_ambiguity_true"],
        0.0,
    )
    output["log1p_summary_ambiguity"] = np.log1p(output["summary_ambiguity"])
    return output


def pairwise_posterior(full_posterior):
    return full_posterior[full_posterior["model"].isin(PAIR_MODELS)].reset_index(drop=True)


def pairwise_posterior_plot(diagnostic, posterior):
    rows = []
    shared = [
        "dataset",
        "id",
        "summary_ambiguity",
        "summary_ambiguity_true",
        "log1p_summary_ambiguity",
        "globally_high_surprise",
        "at_least_one_not_high_surprise",
    ]
    for model in PAIR_MODELS:
        part = diagnostic[shared].copy()
        part["model"] = model
        part["rho"] = diagnostic[f"rho_{model}"]
        part["rho_low"] = diagnostic[f"dm_low_{model}"] / diagnostic[f"dm_high_{model}"]
        rows.append(part)
    distance = pd.concat(rows, ignore_index=True)
    return posterior.merge(distance, on=["dataset", "id", "model"], how="left", validate="one_to_one")


def compute_or_load_pair(config, metric="l2", recompute=False):
    paths = pair_paths(config.summary_label, metric=metric)
    if not recompute and all(path.exists() for path in paths.values()):
        return load_pair_frames(config, metric=metric)

    full = load_cached_all_observed(config=config, metric=metric)
    results = pairwise_results(full["results"])
    diagnostic = pairwise_diagnostic(results, full["diagnostic"])
    posterior = pairwise_posterior(full["posterior"])
    posterior_plot = pairwise_posterior_plot(diagnostic, posterior)
    frames = {
        "results": results,
        "diagnostic": diagnostic,
        "posterior": posterior,
        "posterior_plot": posterior_plot,
    }

    for name, frame in frames.items():
        path = paths[name]
        path.parent.mkdir(parents=True, exist_ok=True)
        frame.to_csv(path, index=False)
    return frames


## Compute Or Load Pair-Specific Results


In [4]:
diagnostics_by_summary = {}

for label, config in summary_configs.items():
    print(f"Computing/loading {label} ({config.summary_label}) for {PAIR_MODELS}")
    diagnostics_by_summary[label] = compute_or_load_pair(
        config=config,
        metric=metric,
        recompute=recompute,
    )


Computing/loading S=D (S1D) for ('m0', 'm2')
Computing/loading S=2D (S2D) for ('m0', 'm2')
Computing/loading S=4D (S4D) for ('m0', 'm2')
Computing/loading S=6D (S6D) for ('m0', 'm2')


## Cached Files


In [5]:
cache_rows = []
for label, config in summary_configs.items():
    for name, path in pair_paths(config.summary_label, metric=metric).items():
        cache_rows.append(
            {
                "summary": label,
                "file_type": name,
                "path": str(path),
                "exists": path.exists(),
            }
        )

cache_files = pd.DataFrame(cache_rows)
cache_files


,summary,file_type,path,exists
0,S=D,results,/Users/yimingzang/Documents/Project/benchmark2...,True
1,S=D,diagnostic,/Users/yimingzang/Documents/Project/benchmark2...,True
2,S=D,posterior,/Users/yimingzang/Documents/Project/benchmark2...,True
3,S=D,posterior_plot,/Users/yimingzang/Documents/Project/benchmark2...,True
4,S=2D,results,/Users/yimingzang/Documents/Project/benchmark2...,True
5,S=2D,diagnostic,/Users/yimingzang/Documents/Project/benchmark2...,True
6,S=2D,posterior,/Users/yimingzang/Documents/Project/benchmark2...,True
7,S=2D,posterior_plot,/Users/yimingzang/Documents/Project/benchmark2...,True
8,S=4D,results,/Users/yimingzang/Documents/Project/benchmark2...,True
9,S=4D,diagnostic,/Users/yimingzang/Documents/Project/benchmark2...,True


## Sanity Checks


In [6]:
coverage_rows = []
for label, frames in diagnostics_by_summary.items():
    for name, frame in frames.items():
        expected_models = set(PAIR_MODELS) if name in {"posterior", "posterior_plot"} else None
        actual_models = set(frame["model"].unique()) if "model" in frame else None
        if expected_models is not None:
            assert actual_models == expected_models, (label, name, actual_models)
        assert set(OBSERVED_DATASETS).issubset(frame["dataset"].unique()), (label, name)
        coverage_rows.append(
            {
                "summary": label,
                "frame": name,
                "rows": len(frame),
                "datasets": ", ".join(frame["dataset"].drop_duplicates()),
                "n_datasets": frame["dataset"].nunique(),
                "models": ", ".join(frame["model"].drop_duplicates()) if "model" in frame else ", ".join(PAIR_MODELS),
            }
        )

    results = frames["results"]
    assert np.allclose(results[[f"pmp_{model}" for model in PAIR_MODELS]].sum(axis=1), 1.0)
    assert np.allclose(results[[f"gold_pmp_{model}" for model in PAIR_MODELS]].sum(axis=1), 1.0)

coverage = pd.DataFrame(coverage_rows)
coverage


,summary,frame,rows,datasets,n_datasets,models
0,S=D,results,136,"empirical, simulated_from_m0, simulated_from_m...",8,"m0, m2"
1,S=D,diagnostic,136,"empirical, simulated_from_m0, simulated_from_m...",8,"m0, m2"
2,S=D,posterior,272,"empirical, simulated_from_m0, simulated_from_m...",8,"m0, m2"
3,S=D,posterior_plot,272,"empirical, simulated_from_m0, simulated_from_m...",8,"m0, m2"
4,S=2D,results,136,"empirical, simulated_from_m0, simulated_from_m...",8,"m0, m2"
5,S=2D,diagnostic,136,"empirical, simulated_from_m0, simulated_from_m...",8,"m0, m2"
6,S=2D,posterior,272,"empirical, simulated_from_m0, simulated_from_m...",8,"m0, m2"
7,S=2D,posterior_plot,272,"empirical, simulated_from_m0, simulated_from_m...",8,"m0, m2"
8,S=4D,results,136,"empirical, simulated_from_m0, simulated_from_m...",8,"m0, m2"
9,S=4D,diagnostic,136,"empirical, simulated_from_m0, simulated_from_m...",8,"m0, m2"


In [7]:
posterior_summary = []
for label, frames in diagnostics_by_summary.items():
    frame = frames["posterior"]
    posterior_summary.append(
        frame.groupby(["dataset", "model"], sort=False)
        .agg(
            mean_mmd=("posterior_mmd", "mean"),
            median_mmd=("posterior_mmd", "median"),
            mean_posterior_mean_rmse=("posterior_mean_rmse", "mean"),
        )
        .assign(summary_dimension=label)
        .reset_index()
    )

posterior_summary = pd.concat(posterior_summary, ignore_index=True)
posterior_summary


,dataset,model,mean_mmd,median_mmd,mean_posterior_mean_rmse,summary_dimension
0,empirical,m0,0.219341,0.154009,0.238050,S=D
1,empirical,m2,0.211853,0.221962,0.220987,S=D
2,simulated_from_m0,m0,0.010768,0.008996,0.040133,S=D
3,simulated_from_m0,m2,0.017084,0.012324,0.045798,S=D
4,simulated_from_m1,m0,0.095185,0.027616,0.203039,S=D
...,...,...,...,...,...,...
59,m3_fast_30,m2,0.138273,0.097240,0.152579,S=6D
60,m3_slow_30,m0,0.211077,0.147073,0.327089,S=6D
61,m3_slow_30,m2,0.110220,0.076350,0.165048,S=6D
62,m3_fast_slow_30,m0,0.281155,0.179708,0.416993,S=6D
